# Custom QDA vs sklearn QDA (Recall Comparison)
This notebook compares recall on the same train/test split and the same preprocessed feature space.

In [1]:
import pandas as pd
import joblib
import __main__
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, f1_score, roc_auc_score, accuracy_score
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from imblearn.over_sampling import SMOTE
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while not (project_root / "app").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.services.custom_qda import CustomQDA

In [2]:
# Load the already-trained custom model artifact
__main__.CustomQDA = CustomQDA
custom_qda = joblib.load('../../analysis/models/custom_qda_model_for_flask.pkl')

In [3]:
df_abt = pd.read_csv('../../data/processed/aml_clean.csv')
df_abt['date'] = pd.to_datetime(df_abt['date']).dt.date
df_abt['time'] = pd.to_datetime(df_abt['time'], format='%H:%M:%S').dt.time
df_abt_cleaned = df_abt.dropna(subset=['is_laundering'])

key_vars = [
    'date', 'time', 's_bank_name', 's_bank_id', 's_entity_id',
    's_entity_name', 's_entity_type', 's_latitude', 's_longitude',
    'r_bank_name', 'r_bank_id', 'r_entity_id', 'r_entity_name',
    'r_entity_type', 'r_latitude', 'r_longitude'
]
target = 'is_laundering'

subset_proportion = 0.1
if subset_proportion < 1.0:
    df_abt_subset, _ = train_test_split(
        df_abt_cleaned,
        train_size=subset_proportion,
        random_state=42,
        stratify=df_abt_cleaned[target]
    )
else:
    df_abt_subset = df_abt_cleaned.copy()

train_df, test_df = train_test_split(
    df_abt_subset,
    test_size=0.2,
    random_state=42,
    stratify=df_abt_subset[target]
)

X_train_raw = train_df.drop(columns=key_vars + [target])
y_train = train_df[target]
X_test_raw = test_df.drop(columns=key_vars + [target])
y_test = test_df[target]

loaded_preprocessing_pipeline = joblib.load('../../analysis/features/pycaret_preprocessing_pipeline.pkl')
X_train_preprocessed = loaded_preprocessing_pipeline.transform(X_train_raw)
X_test_preprocessed = loaded_preprocessing_pipeline.transform(X_test_raw)

smote_instance = SMOTE(k_neighbors=3, random_state=42, sampling_strategy=0.2)
X_train_smoted, y_train_smoted = smote_instance.fit_resample(X_train_preprocessed, y_train)

In [4]:
# Train only sklearn QDA on the same prepared training data
sk_qda = QuadraticDiscriminantAnalysis(reg_param=0.14, store_covariance=True)
sk_qda.fit(X_train_smoted, y_train_smoted)

QuadraticDiscriminantAnalysis(reg_param=0.14, store_covariance=True)

In [6]:
# CustomQDA evaluation
y_pred_custom = custom_qda.predict(X_test_preprocessed)
y_proba_custom = custom_qda.predict_proba(X_test_preprocessed)[:, 1]

custom_results = {
    'model': 'CustomQDA',
    'recall': recall_score(y_test, y_pred_custom),
    'f1': f1_score(y_test, y_pred_custom),
    'auc': roc_auc_score(y_test, y_proba_custom),
    'accuracy': accuracy_score(y_test, y_pred_custom)
}

# sklearn QDA evaluation
y_pred_sklearn = sk_qda.predict(X_test_preprocessed)
y_proba_sklearn = sk_qda.predict_proba(X_test_preprocessed)[:, 1]

sklearn_results = {
    'model': 'sklearn QDA',
    'recall': recall_score(y_test, y_pred_sklearn),
    'f1': f1_score(y_test, y_pred_sklearn),
    'auc': roc_auc_score(y_test, y_proba_sklearn),
    'accuracy': accuracy_score(y_test, y_pred_sklearn)
}

results_df = pd.DataFrame([custom_results, sklearn_results]).sort_values(by='recall', ascending=False).reset_index(drop=True)
results_df

,model,recall,f1,auc,accuracy
0,CustomQDA,0.855769,0.014542,0.920181,0.881243
1,sklearn QDA,0.855769,0.014542,0.920182,0.881243
